# CPSC 483 — Day 5 Class Work #1
## SVM: Classifying Iris setosa vs. virginica
**Using only Petal Length and Petal Width**

Goals:
1. Train a linear SVM on setosa vs. virginica (ignoring versicolor)
2. Report the **equation of the decision boundary**
3. Report **how many support vectors** are needed

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.svm import SVC

## 2. Load data — petal features only, drop versicolor

In [ ]:
iris = load_iris()

# Features: columns 2 and 3 = petal length, petal width
X = iris.data[:, 2:4]
y = iris.target          # 0=setosa, 1=versicolor, 2=virginica

# Drop versicolor (label == 1)
mask = y != 1
X, y = X[mask], y[mask]

print("Classes remaining:", np.unique(y), "→", [iris.target_names[c] for c in np.unique(y)])
print("Dataset shape:", X.shape)

## 3. Train the SVM

`C=1e100` sets an astronomically high penalty for any margin violation — effectively enforcing a **hard margin** (no point is allowed to fall inside or on the wrong side). These two classes are perfectly linearly separable in petal space, so the hard margin has a solution.

In [ ]:
svm = SVC(kernel="linear", C=1e100)
svm.fit(X, y)

## 4. Decision boundary equation

For a linear SVM the decision surface is `w · x + b = 0`, i.e.:

```
w[0] * petal_length + w[1] * petal_width + b = 0
```

Solving for `petal_width` gives the boundary line we can plot:

```
petal_width = -(w[0] * petal_length + b) / w[1]
```

In [ ]:
w = svm.coef_[0]          # weight vector  [w0, w1]
b = svm.intercept_[0]     # bias term

print("=" * 50)
print("DECISION BOUNDARY EQUATION")
print("=" * 50)
print(f"  {w[0]:.4f} * petal_length  +  {w[1]:.4f} * petal_width  +  ({b:.4f})  =  0")
print()
print("Solved for petal_width:")
print(f"  petal_width  =  -({w[0]:.4f} * petal_length  +  ({b:.4f}))  /  {w[1]:.4f}")
print()
# Simplified slope-intercept form:  petal_width = m * petal_length + c
m = -w[0] / w[1]
c = -b / w[1]
print(f"  petal_width  =  {m:.4f} * petal_length  +  {c:.4f}")

## 5. Support vectors

In [ ]:
print("=" * 50)
print("SUPPORT VECTORS")
print("=" * 50)
print(f"Total support vectors : {len(svm.support_vectors_)}")
print(f"Per class (setosa / virginica) : {svm.n_support_}")
print()
print("Support vector coordinates (petal_length, petal_width):")
for i, sv in enumerate(svm.support_vectors_):
    print(f"  SV {i+1}: petal_length={sv[0]:.1f},  petal_width={sv[1]:.1f}")

## 6. Visualization — decision boundary + margin + support vectors

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = {0: "steelblue", 2: "tomato"}
labels = {0: "setosa (0)", 2: "virginica (2)"}

for cls in [0, 2]:
    idx = y == cls
    ax.scatter(X[idx, 0], X[idx, 1],
               c=colors[cls], label=labels[cls],
               edgecolors="k", linewidths=0.5, s=60, zorder=3)

# Decision boundary and margin lines over the petal-length range
x_range = np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 300)

def boundary_line(x_vals, offset=0):
    """petal_width for w·x + b = offset"""
    return -(w[0] * x_vals + b - offset) / w[1]

ax.plot(x_range, boundary_line(x_range, 0),  "k-",  lw=2,   label="Decision boundary")
ax.plot(x_range, boundary_line(x_range, +1), "k--", lw=1.2, label="Margin edge (+1)")
ax.plot(x_range, boundary_line(x_range, -1), "k--", lw=1.2, label="Margin edge (−1)")

# Highlight support vectors
ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
           s=200, facecolors="none", edgecolors="gold",
           linewidths=2, zorder=4, label="Support vectors")

ax.set_xlabel("Petal Length (cm)", fontsize=12)
ax.set_ylabel("Petal Width (cm)", fontsize=12)
ax.set_title("Linear SVM — Setosa vs. Virginica (petal features)", fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(x_range[0], x_range[-1])
plt.tight_layout()
plt.show()

## 7. Summary

Run the cell below for a single printed summary — the answer to both classwork questions.

In [ ]:
print("━" * 55)
print("CLASS WORK #1 — ANSWERS")
print("━" * 55)
print()
print("Q1 — Decision boundary equation:")
print(f"  {w[0]:.4f}·petal_length + {w[1]:.4f}·petal_width + ({b:.4f}) = 0")
print(f"  → petal_width = {m:.4f}·petal_length + {c:.4f}")
print()
print("Q2 — Support vectors:")
print(f"  {len(svm.support_vectors_)} total  "
      f"({svm.n_support_[0]} from setosa, {svm.n_support_[1]} from virginica)")
print()
print("Note: setosa and virginica are perfectly separable in petal")
print("space, so only a tiny number of points sit on the margin")
print("edges — everything else is irrelevant to the boundary.")
print("━" * 55)